# 05 — Features, explicit labels, chronological splits, and robustness

For local series $s$, define $P_{s,t}$ as the observed weekly median:

$$y_{s,t}=\mathbf{1}\left[100(P_{s,t+1}/P_{s,t}-1)\geq10\right].$$

The 10% threshold is a fixed working research definition, not an official standard or
an optimised economic cutoff. Compare 5%, 10%, and 15% on training data only.
Predict **after current-week observations are available**; release delays are a limitation.
Both weeks must meet coverage requirements. Missing outcomes remain unlabelled.
Missing historical prices remain missing; no interpolation or backward fill is used.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Run from the repository root or its notebooks folder.")
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import numpy as np
import pricecatcher_config as cfg
from data_utils import save_json, surge_event
from data_utils import make_features, time_masks
output = cfg.stage(5)
weekly = pd.read_parquet(cfg.stage(4) / "weekly.parquet")
featured = make_features(weekly)
# Store all series here. Notebook 06 learns item inclusion independently in each fold.
featured.to_parquet(output / "features.parquet", index=False)
train_mask, validation_mask = time_masks(featured, cfg.TRAIN_END, cfg.VALIDATION_END)
_, test_mask = time_masks(featured, cfg.VALIDATION_END, pd.Period(cfg.END_MONTH).end_time)
splits = {"train": featured.loc[train_mask].copy(), "validation": featured.loc[validation_mask].copy(),
          "test": featured.loc[test_mask].copy()}
summary = []
for name, frame in splits.items():
    if frame.empty:
        raise ValueError(f"{name} is empty; check dates and coverage.")
    frame.to_parquet(output / f"{name}.parquet", index=False)
    summary.append({"split": name, "rows": len(frame), "weeks": frame["week_start"].nunique(),
                    "surges": int(frame["surge_label"].sum()), "positive_rate": float(frame["surge_label"].mean())})
pd.DataFrame(summary).to_csv(output / "split_summary.csv", index=False)
display(pd.DataFrame(summary))
# Data available for a real next-week forecast; never require future coverage at inference.
candidates = featured.loc[featured["current_coverage_ok"] & featured["history_ok"]].copy()
candidates.to_parquet(output / "forecast_candidates.parquet", index=False)
assert not any(c.startswith("next_") or c in ["surge_label", "common_panel_change_pct"] for c in cfg.FEATURES)


,split,rows,weeks,surges,positive_rate
0,train,220108,48,26525,0.120509
1,validation,50827,12,5014,0.098648
2,test,62613,12,9009,0.143884


## Training-only sensitivity: thresholds, coverage, and aggregation
The matched-premise comparison uses the same premises in both weeks. It gives a sensitivity
result on a potentially smaller population, not a replacement ground truth. Its future
premise overlap cannot determine whether a live forecast is issued.
Inspect large disagreements before interpreting a model as predicting within-shop inflation.

In [2]:
train = splits["train"]
threshold_rows = []
for group, part in [("ALL", train)] + list(train.groupby("project_group")):
    for threshold in [5.0, 10.0, 15.0]:
        threshold_rows.append({"project_group": group, "threshold_pct": threshold,
            "rows": len(part), "positives": int(surge_event(part["next_week_change_pct"], threshold).sum())})
pd.DataFrame(threshold_rows).to_csv(output / "training_threshold_sensitivity.csv", index=False)
display(pd.DataFrame(threshold_rows))

equal_change = (train["next_equal_premise_median"] / train["equal_premise_median"] - 1) * 100
aggregation_rows = []
for name, changes in [("daily_observation_median", train["next_week_change_pct"]),
                       ("equal_premise_median", equal_change),
                       ("matched_premises", train["common_panel_change_pct"])]:
    supported = changes.notna()
    alternative = surge_event(changes.loc[supported]).astype(int)
    primary = train.loc[supported, "surge_label"].astype(int)
    aggregation_rows.append({"aggregation": name, "supported_rows": int(supported.sum()),
        "surges": int(alternative.sum()),
        "label_disagreement_rate": float(alternative.ne(primary).mean()) if len(primary) else None})
aggregation_sensitivity = pd.DataFrame(aggregation_rows)
aggregation_sensitivity.to_csv(output / "training_aggregation_sensitivity.csv", index=False)
display(aggregation_sensitivity)
train.nlargest(30, "next_week_change_pct").to_csv(output / "extreme_training_changes.csv", index=False)

coverage_rows = []
training_period = featured.loc[featured["next_week_end"] <= pd.Timestamp(cfg.TRAIN_END)]
for minimum_premises in [1, 2, 3]:
    for minimum_records in [2, 4]:
        keep = (training_period["record_count"].ge(minimum_records)
            & training_period["next_week_record_count"].ge(minimum_records)
            & training_period["premise_count"].ge(minimum_premises)
            & training_period["next_week_premise_count"].ge(minimum_premises)
            & training_period["history_ok"] & training_period["next_week_price"].notna())
        part = training_period.loc[keep]
        coverage_rows.append({"minimum_premises": minimum_premises, "minimum_records": minimum_records,
            "rows": len(part), "state_district_pairs": len(part[["state", "district"]].drop_duplicates()),
            "surges": int(surge_event(part["next_week_change_pct"]).sum())})
pd.DataFrame(coverage_rows).to_csv(output / "training_coverage_sensitivity.csv", index=False)

audit = training_period.copy()
audit["eligible"] = audit["surge_label"].notna() & audit["history_ok"]
audit.groupby(["state", "district"]).agg(observed_weeks=("week_start", "size"),
    eligible_rows=("eligible", "sum"), eligibility_rate=("eligible", "mean")).to_csv(output / "training_district_eligibility.csv")
save_json(output / "experiment_settings.json", {
    "surge_threshold_pct": cfg.SURGE_THRESHOLD_PCT, "minimum_records": cfg.MIN_RECORDS,
    "minimum_premises": cfg.MIN_PREMISES, "minimum_history_4w": cfg.MIN_HISTORY_4W,
    "train_end": cfg.TRAIN_END, "validation_end": cfg.VALIDATION_END,
    "features": cfg.FEATURES, "categorical_features": cfg.CAT_FEATURES,
    "numeric_features": cfg.NUM_FEATURES,
    "target": "next calendar week's observed median rises at least 10%; not necessarily within-premise inflation",
    "missing_policy": "no price filling; model numeric imputation fitted on each training fold",
    "prediction_time": "after current week observations are available",
})


,project_group,threshold_pct,rows,positives
0,ALL,5.0,220108,38309
1,ALL,10.0,220108,26525
2,ALL,15.0,220108,17574
3,Chicken,5.0,19732,1321
4,Chicken,10.0,19732,304
5,Chicken,15.0,19732,47
6,Cooking Oil,5.0,2568,1
7,Cooking Oil,10.0,2568,0
8,Cooking Oil,15.0,2568,0
9,Eggs,5.0,40195,4029


,aggregation,supported_rows,surges,label_disagreement_rate
0,daily_observation_median,220108,26525,0.000000
1,equal_premise_median,220108,25846,0.049526
2,matched_premises,219873,23999,0.050757


## Handoff to Notebook 06
The exported splits include diagnostics for research, but only `cfg.FEATURES` may enter X.
Fit encoding, imputation, and class weighting inside each training fold. Training outcomes
must end before evaluation predictions, including the one-week horizon. Use validation
for the alert cutoff and test for one final assessment. Unsupported district/group coverage
and absence of positive events must be reported separately from successful prediction.